# TCN v2.5 학습 노트북

이 노트북은 `Colab`에서 바로 실행할 수 있도록 정리한 `TCN v2.5` 학습 파이프라인입니다.

## 포함 내용
- Google Drive 마운트
- 프로젝트 루트, 데이터 경로, 아티팩트 경로 환경변수 설정
- `CSV` 기본 / `SQLite` 선택 가능 데이터 로더
- `video_id` 기준 그룹 분할
- `TCN v2.5` 슬라이딩 윈도우 학습
- 학습 곡선 시각화
- 혼동행렬, 분류 리포트, 핵심 지표 출력
- 예시 비디오의 시간축 확률 곡선 시각화

현재 기본 설정은 `4~10초` monitoring 구간 안에서 `60-frame` 슬라이딩 윈도우를 여러 개 생성해 학습합니다.


In [ ]:
# Colab에서만 Google Drive를 마운트합니다.
import os

IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive 마운트 완료: /content/drive')
else:
    print('Colab 환경이 아니므로 Drive 마운트를 건너뜁니다.')

## 1. 의존성 설치

런타임을 새로 시작했거나 패키지가 없다면 아래 셀을 먼저 실행하세요.


In [ ]:
!pip install -q pandas numpy scikit-learn tensorflow matplotlib seaborn

## 2. 프로젝트 경로 및 환경변수 설정

아래 셀에서 `PROJECT_ROOT`를 실제 저장소 위치에 맞게 잡습니다.

- Colab + Drive에서 실행하면 `PROJECT_ROOT_CANDIDATES`를 순서대로 확인합니다.
- 원하는 경로가 있으면 `MANUAL_PROJECT_ROOT`에 직접 넣어도 됩니다.
- 최종적으로 `PROJECT_ROOT`, `DATA_ROOT`, `ARTIFACT_ROOT` 환경변수를 설정합니다.


In [ ]:
from pathlib import Path
import sys
import os

MANUAL_PROJECT_ROOT = ''
PROJECT_ROOT_CANDIDATES = [
    '/content/drive/MyDrive/Graduate-Project/Falling-Model-Development',
    '/content/drive/MyDrive/졸업 과제/Falling-Model-Development',
    '/content/Falling-Model-Development',
    str(Path.cwd()),
]

candidate_paths = [MANUAL_PROJECT_ROOT] + PROJECT_ROOT_CANDIDATES if MANUAL_PROJECT_ROOT else PROJECT_ROOT_CANDIDATES
PROJECT_ROOT = None
for candidate in candidate_paths:
    path = Path(candidate)
    if (path / 'scripts').exists() and (path / 'dataset').exists():
        PROJECT_ROOT = path.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다. MANUAL_PROJECT_ROOT를 직접 지정하세요.')

DATA_ROOT = PROJECT_ROOT / 'dataset'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'tcn_v25_notebook'

os.environ['PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['DATA_ROOT'] = str(DATA_ROOT)
os.environ['ARTIFACT_ROOT'] = str(ARTIFACT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('ARTIFACT_ROOT =', ARTIFACT_ROOT)

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from IPython.display import display

mpl.rcParams['font.family'] = ['DejaVu Sans', 'sans-serif']
mpl.rcParams['axes.unicode_minus'] = False

from scripts.train_tcn_v2 import (
    build_tcn_v2_model,
    class_weight_from_labels,
    compile_model,
    evaluate_model,
    export_tflite_artifacts,
    load_source_frame,
    log,
    make_tf_dataset,
    normalize_splits,
    save_run_metadata,
    save_visualization_artifacts,
    train_model,
    set_seed,
)
from scripts.train_tcn_v25 import (
    TcnV25TrainConfig,
    build_monitoring_segments,
    build_sliding_window_dataset,
    describe_window_split,
    parse_int_list,
    split_segments,
)

## 3. 실험 설정

아래 값만 바꾸면 됩니다.

- `INPUT_FORMAT`: `csv` 또는 `sqlite`
- `CSV_PATH`, `SQLITE_PATH`: 입력 데이터 경로
- `TARGET_STEPS`: 최종 입력 길이
- `DILATIONS`, `CHANNELS`: TCN 블록 구조
- `EXPORT_TFLITE`: `True`로 바꾸면 학습 후 `fp32/int8 tflite`까지 내보냅니다.


In [ ]:
INPUT_FORMAT = 'csv'
CSV_PATH = str(DATA_ROOT / 'final_dataset.csv')
SQLITE_PATH = str(DATA_ROOT / 'final_dataset.sqlite')
SQLITE_TABLE = 'fall_frames'

OUTPUT_DIR = str(ARTIFACT_ROOT)
MONITOR_START_SEC = 4.0
MONITOR_END_SEC = 10.0
TARGET_STEPS = 60
LABEL_MODE = 'last_frame'

BATCH_SIZE = 128
EPOCHS = 30
LEARNING_RATE = 1e-3
RANDOM_STATE = 42

KERNEL_SIZE = 3
DROPOUT_RATE = 0.15
DILATIONS = '1,2,4,8'
CHANNELS = '32,32,64,96'
TRAIN_POSITIVE_STRIDE = 1
TRAIN_NEGATIVE_STRIDE = 5
EVAL_STRIDE = 1
EXPORT_TFLITE = False

config = TcnV25TrainConfig(
    input_format=INPUT_FORMAT,
    csv_path=CSV_PATH,
    sqlite_path=SQLITE_PATH,
    sqlite_table=SQLITE_TABLE,
    output_dir=OUTPUT_DIR,
    monitor_start_sec=MONITOR_START_SEC,
    monitor_end_sec=MONITOR_END_SEC,
    target_steps=TARGET_STEPS,
    label_mode=LABEL_MODE,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    random_state=RANDOM_STATE,
    kernel_size=KERNEL_SIZE,
    dropout_rate=DROPOUT_RATE,
    dilations=parse_int_list(DILATIONS),
    channels=parse_int_list(CHANNELS),
    export_tflite=EXPORT_TFLITE,
    train_positive_stride=TRAIN_POSITIVE_STRIDE,
    train_negative_stride=TRAIN_NEGATIVE_STRIDE,
    eval_stride=EVAL_STRIDE,
)

display(pd.DataFrame([config.__dict__]))

In [ ]:

from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score


def plot_history(history):
    history_df = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    history_df[['loss', 'val_loss']].plot(ax=axes[0], marker='o')
    axes[0].set_title('학습 손실 곡선')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)

    metric_cols = [col for col in ['accuracy', 'val_accuracy', 'recall', 'val_recall', 'precision', 'val_precision'] if col in history_df.columns]
    history_df[metric_cols].plot(ax=axes[1], marker='o')
    axes[1].set_title('성능 지표 곡선')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Score')
    axes[1].grid(True, alpha=0.3)
    plt.show()


def plot_confusion(metrics, split_name, cmap='Blues'):
    cm = np.array(metrics['confusion_matrix'])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, cbar=False)
    plt.title(f'{split_name} confusion matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()


def metrics_table(metrics_dict):
    rows = []
    for split_name, metric in metrics_dict.items():
        rows.append({
            'split': split_name,
            'accuracy': metric['accuracy'],
            'precision': metric['precision'],
            'recall': metric['recall'],
            'macro_f1': metric['macro_f1'],
        })
    return pd.DataFrame(rows)


def apply_consecutive_rule(binary_pred, min_consecutive=1):
    binary_pred = np.asarray(binary_pred, dtype=np.int32)
    if min_consecutive <= 1:
        return binary_pred

    filtered = np.zeros_like(binary_pred)
    start = None
    for idx, value in enumerate(binary_pred):
        if value == 1 and start is None:
            start = idx
        elif value == 0 and start is not None:
            if idx - start >= min_consecutive:
                filtered[start:idx] = 1
            start = None
    if start is not None and len(binary_pred) - start >= min_consecutive:
        filtered[start:] = 1
    return filtered


def evaluate_threshold_predictions(y_true, positive_scores, threshold, min_consecutive=1):
    raw_pred = (np.asarray(positive_scores) >= threshold).astype(np.int32)
    pred = apply_consecutive_rule(raw_pred, min_consecutive=min_consecutive)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'accuracy': float((pred == y_true).mean()),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'macro_f1': float(f1_score(y_true, pred, average='macro', zero_division=0)),
        'confusion_matrix': cm.tolist(),
        'classification_report': classification_report(y_true, pred, digits=4, zero_division=0),
        'threshold': float(threshold),
        'min_consecutive': int(min_consecutive),
        'predicted_positive_rate': float(pred.mean()),
    }


def threshold_sweep(y_true, positive_scores, thresholds=None, min_consecutive=1):
    thresholds = np.linspace(0.05, 0.95, 19) if thresholds is None else np.asarray(thresholds, dtype=np.float32)
    rows = []
    for threshold in thresholds:
        metrics = evaluate_threshold_predictions(y_true, positive_scores, float(threshold), min_consecutive=min_consecutive)
        rows.append({
            'threshold': float(threshold),
            'accuracy': metrics['accuracy'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'macro_f1': metrics['macro_f1'],
            'predicted_positive_rate': metrics['predicted_positive_rate'],
            'min_consecutive': int(min_consecutive),
        })
    return pd.DataFrame(rows)


def select_best_threshold(sweep_df, metric='macro_f1', min_precision=None, min_recall=None):
    filtered = sweep_df.copy()
    if min_precision is not None:
        filtered = filtered[filtered['precision'] >= min_precision]
    if min_recall is not None:
        filtered = filtered[filtered['recall'] >= min_recall]
    if filtered.empty:
        filtered = sweep_df.copy()
    return filtered.sort_values([metric, 'precision', 'recall', 'threshold'], ascending=[False, False, False, False]).iloc[0]


def plot_threshold_sweep(sweep_df, title):
    plt.figure(figsize=(10, 4))
    plt.plot(sweep_df['threshold'], sweep_df['precision'], marker='o', label='precision')
    plt.plot(sweep_df['threshold'], sweep_df['recall'], marker='o', label='recall')
    plt.plot(sweep_df['threshold'], sweep_df['macro_f1'], marker='o', label='macro_f1')
    plt.title(title)
    plt.xlabel('threshold')
    plt.ylabel('score')
    plt.ylim(0.0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


def load_video_frame_df(config, video_id, feature_cols):
    if config.input_format == 'csv':
        df = pd.read_csv(config.csv_path)
        df = df[df['video_id'] == video_id].sort_values(['time_sec', 'frame']).reset_index(drop=True)
        keep_cols = ['video_id', 'frame', 'time_sec', 'label'] + feature_cols
        return df[keep_cols]

    with sqlite3.connect(config.sqlite_path) as conn:
        select_cols = ['video_id', 'frame', 'time_sec', 'label'] + feature_cols
        quoted_cols = ', '.join(f'"{col}"' for col in select_cols)
        query = f'''
            SELECT {quoted_cols}
            FROM "{config.sqlite_table}"
            WHERE video_id = ?
            ORDER BY time_sec, frame
        '''
        return pd.read_sql_query(query, conn, params=[video_id])


def make_sliding_windows(video_df, feature_cols, window_size):
    values = video_df[feature_cols].to_numpy(dtype=np.float32)
    labels = video_df['label'].to_numpy(dtype=np.int32)
    times = video_df['time_sec'].to_numpy(dtype=np.float32)

    if len(values) < window_size:
        return None, None, None

    windows = []
    targets = []
    target_times = []
    for idx in range(len(values) - window_size + 1):
        windows.append(values[idx: idx + window_size])
        targets.append(labels[idx + window_size - 1])
        target_times.append(times[idx + window_size - 1])

    return np.asarray(windows, dtype=np.float32), np.asarray(targets), np.asarray(target_times)


def plot_video_probability_examples(model, config, feature_cols, mean, std, video_ids, window_size=60, threshold=0.5, min_consecutive=1):
    for video_id in video_ids:
        video_df = load_video_frame_df(config, video_id, feature_cols)
        windows, targets, target_times = make_sliding_windows(video_df, feature_cols, window_size)
        if windows is None:
            print(f'{video_id}: skipped because frames are insufficient for one window')
            continue

        windows = ((windows - mean) / std).astype(np.float32)
        probs = model.predict(windows, verbose=0)[:, 1]
        alarm = apply_consecutive_rule((probs >= threshold).astype(np.int32), min_consecutive=min_consecutive)

        plt.figure(figsize=(12, 4))
        plt.plot(target_times, probs, color='crimson', label='fall probability')
        plt.fill_between(target_times, targets, alpha=0.2, color='gray', label='ground truth')
        plt.fill_between(target_times, alarm, alpha=0.15, color='orange', label='alarm after rule')
        plt.axhline(threshold, color='royalblue', linestyle='--', label=f'threshold={threshold:.2f}')
        plt.ylim(-0.05, 1.05)
        plt.title(f'Video probability curve: {video_id}')
        plt.xlabel('time_sec')
        plt.ylabel('fall probability')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()


## 4. 슬라이딩 윈도우 데이터 준비 및 모델 생성

이 단계에서 입력 데이터를 읽고, 클립을 구성하고, 학습/검증/테스트 셋으로 분리한 뒤 모델을 만듭니다.


In [ ]:
set_seed(config.random_state)
log(
    'tcn v2.5 notebook start '
    f'input_format={config.input_format} target_steps={config.target_steps} '
    f'dilations={config.dilations} channels={config.channels} '
    f'strides(train_pos={config.train_positive_stride}, train_neg={config.train_negative_stride}, eval={config.eval_stride})'
)

df, feature_cols = load_source_frame(config)
segments = build_monitoring_segments(
    df=df,
    feature_cols=feature_cols,
    monitor_start_sec=config.monitor_start_sec,
    monitor_end_sec=config.monitor_end_sec,
)
segment_positive_count = int(sum(segment['segment_label'] for segment in segments))
segment_summary_df = pd.DataFrame({
    'segment_count': [len(segments)],
    'feature_count': [len(feature_cols)],
    'positive_videos': [segment_positive_count],
    'negative_videos': [int(len(segments) - segment_positive_count)],
})
display(segment_summary_df)

split_segments_map = split_segments(segments, config.random_state)
x_train, y_train, group_train = build_sliding_window_dataset(
    split_segments_map['train'],
    config.target_steps,
    config.label_mode,
    training=True,
    positive_stride=config.train_positive_stride,
    negative_stride=config.train_negative_stride,
    eval_stride=config.eval_stride,
)
x_val, y_val, group_val = build_sliding_window_dataset(
    split_segments_map['val'],
    config.target_steps,
    config.label_mode,
    training=False,
    positive_stride=config.train_positive_stride,
    negative_stride=config.train_negative_stride,
    eval_stride=config.eval_stride,
)
x_test, y_test, group_test = build_sliding_window_dataset(
    split_segments_map['test'],
    config.target_steps,
    config.label_mode,
    training=False,
    positive_stride=config.train_positive_stride,
    negative_stride=config.train_negative_stride,
    eval_stride=config.eval_stride,
)

print(describe_window_split('train', y_train, group_train))
print(describe_window_split('val', y_val, group_val))
print(describe_window_split('test', y_test, group_test))

x_train, x_val, x_test, mean, std = normalize_splits(x_train, x_val, x_test)
split_sizes = {
    'train_windows': int(len(y_train)),
    'val_windows': int(len(y_val)),
    'test_windows': int(len(y_test)),
    'train_videos': int(len(np.unique(group_train))),
    'val_videos': int(len(np.unique(group_val))),
    'test_videos': int(len(np.unique(group_test))),
}
display(pd.DataFrame([split_sizes]))

train_ds = make_tf_dataset(x_train, y_train, config.batch_size, training=True)
val_ds = make_tf_dataset(x_val, y_val, config.batch_size, training=False)
class_weight = class_weight_from_labels(y_train)
print('class_weight =', class_weight)

model = build_tcn_v2_model(config, input_shape=(config.target_steps, len(feature_cols)))
model = compile_model(model, config.learning_rate)
model.summary()

## 5. 모델 학습

학습 중에는 `EarlyStopping`과 `ReduceLROnPlateau`가 함께 동작합니다.


In [ ]:
history = train_model(model, train_ds, val_ds, config.epochs, class_weight)

## 6. 평가, 저장, 시각화

학습이 끝나면 모델 저장, 메타데이터 저장, 지표 출력, 혼동행렬/학습 곡선을 시각화합니다.


In [ ]:

output_dir = Path(config.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

keras_path = output_dir / 'tcn_v25.keras'
model.save(keras_path)
log(f'saved keras model to {keras_path}')

train_prob = model.predict(x_train, verbose=0)[:, 1]
val_prob = model.predict(x_val, verbose=0)[:, 1]
test_prob = model.predict(x_test, verbose=0)[:, 1]

metrics = {
    'train': evaluate_model(model, x_train, y_train, 'train'),
    'val': evaluate_model(model, x_val, y_val, 'val'),
    'test': evaluate_model(model, x_test, y_test, 'test'),
}

THRESHOLD_GRID = np.linspace(0.05, 0.95, 19)
MIN_CONSECUTIVE = 3
threshold_sweep_df = threshold_sweep(y_val, val_prob, thresholds=THRESHOLD_GRID, min_consecutive=MIN_CONSECUTIVE)
best_threshold_row = select_best_threshold(threshold_sweep_df, metric='macro_f1')
selected_threshold = float(best_threshold_row['threshold'])

threshold_metrics = {
    'train_thresholded': evaluate_threshold_predictions(y_train, train_prob, selected_threshold, min_consecutive=MIN_CONSECUTIVE),
    'val_thresholded': evaluate_threshold_predictions(y_val, val_prob, selected_threshold, min_consecutive=MIN_CONSECUTIVE),
    'test_thresholded': evaluate_threshold_predictions(y_test, test_prob, selected_threshold, min_consecutive=MIN_CONSECUTIVE),
}

export_paths = {'keras': str(keras_path)}
if config.export_tflite:
    log('tflite export start')
    export_paths.update(export_tflite_artifacts(model, x_train, output_dir, 'tcn_v25'))
    log('tflite export done')
export_paths.update(save_visualization_artifacts(output_dir, history, metrics))

metadata_path = save_run_metadata(
    config=config,
    feature_cols=feature_cols,
    mean=mean,
    std=std,
    split_sizes=split_sizes,
    metrics={**metrics, **threshold_metrics},
    history=history,
    export_paths=export_paths,
)

display(metrics_table(metrics))
plot_history(history)
plot_confusion(metrics['test'], 'test')

print('selected threshold =', round(selected_threshold, 4), '| min_consecutive =', MIN_CONSECUTIVE)
display(threshold_sweep_df.sort_values('macro_f1', ascending=False).head(10))
plot_threshold_sweep(threshold_sweep_df, 'Validation threshold sweep')

display(metrics_table(threshold_metrics))
plot_confusion(threshold_metrics['test_thresholded'], 'test thresholded')

print('metadata path:', metadata_path)
print('\n[test argmax classification report]')
print(metrics['test']['classification_report'])
print('test argmax confusion matrix:', metrics['test']['confusion_matrix'])
print('\n[test thresholded classification report]')
print(threshold_metrics['test_thresholded']['classification_report'])
print('test thresholded confusion matrix:', threshold_metrics['test_thresholded']['confusion_matrix'])


## 7. 예시 비디오 시각화

`tcn_v2.5` 학습 후 실제 비디오별 시간축 확률 곡선을 확인할 수 있는 셀입니다.

- 테스트 셋에서 양성/음성 비디오를 각각 일부 골라서 보여줍니다.
- `window_size`는 기본적으로 `60`입니다.


In [ ]:

test_video_df = pd.DataFrame({
    'video_id': group_test,
    'label': y_test,
})

positive_video_ids = test_video_df[test_video_df['label'] == 1]['video_id'].unique().tolist()[:2]
negative_video_ids = test_video_df[test_video_df['label'] == 0]['video_id'].unique().tolist()[:2]
example_video_ids = positive_video_ids + negative_video_ids

print('example video_id:', example_video_ids)
print('applied threshold =', round(selected_threshold, 4), '| min_consecutive =', MIN_CONSECUTIVE)
plot_video_probability_examples(
    model=model,
    config=config,
    feature_cols=feature_cols,
    mean=mean,
    std=std,
    video_ids=example_video_ids,
    window_size=config.target_steps,
    threshold=selected_threshold,
    min_consecutive=MIN_CONSECUTIVE,
)
